# Production RAG operations

A production RAG service needs traces, budgets, freshness checks, and readiness signals. This notebook builds those controls as small, testable policies.

## Operational loop

```mermaid
flowchart LR
 Q[Request] --> T[Trace]
 T --> B{Budget}
 B -->|pass| R[Response]
 B -->|fail| F[Fallback]
 I[Index freshness] --> H[Readiness]
 E[Evaluation health] --> H
```

In [ ]:
from datetime import datetime, timedelta, timezone
from examples.advanced.operations import Budget, Trace, freshness_status, health_status, within_budget

trace = Trace('How does RAG work?', 'retrieve')
trace.record('retrieval-start')
trace.latency_ms, trace.cost_usd = 180, 0.002
within_budget(trace, Budget(500, 0.01)), trace.events

In [ ]:
now = datetime.now(timezone.utc)
freshness_status(now - timedelta(hours=2), now, 24), freshness_status(now - timedelta(hours=48), now, 24)
health_status(index_ready=True, evaluator_ready=True, corpus_fresh=False)

## Exercise

Add a trace ID and circuit breaker. Simulate a latency budget breach and a stale index, then define whether the service should fail closed, serve a cached response, or escalate.